# YOLO11 NSFW Detection Training - Kaggle Notebook
## Safe Training with No Image Display

This notebook trains a YOLO11 model for NSFW body-part detection.
- **NEVER displays images** (NSFW safety)
- All outputs saved to `/kaggle/working/outputs`
- Final ZIP created for download

##  Resuming Training After Session Restart

**To resume training after restarting Kaggle session:**

1. **Download checkpoints** from previous run:
   - `last.pt` (preferred - contains optimizer state)
   - `best.pt` (alternative)

2. **Upload checkpoints** to Kaggle Input Dataset:
   - Go to "Add Data" → "Upload" in Kaggle notebook
   - Upload `last.pt` or `best.pt` file
   - Or create a dataset with structure:
     ```
     your_dataset/
       ├── last.pt
       └── best.pt
     ```
   - Or upload to a subdirectory:
     ```
     your_dataset/
       ├── weights/
       │   ├── last.pt
       │   └── best.pt
     ```

3. **Run the notebook** - it will automatically detect and resume from the checkpoint!

The code searches for checkpoints in:
- `/kaggle/input/` (persistent - your uploaded datasets) ← **PRIORITY**
- `/kaggle/working/runs/` (current session only)


## 1. Setup: Download and Extract Dataset


In [ ]:
# Dataset configuration
DIRECT_URL = "https://drive.google.com/uc?export=download&id=1XFTw-fmO6PWlsVv-6Hh5AkQeVhsY8wa5"
ARCHIVE_PATH = "/kaggle/working/final_balanced.7z"
DATA_DIR_SOURCE = "/kaggle/working/final_balanced"
# Dataset directory - actual nested path after extraction
DATA_DIR = "/kaggle/working/final_balanced/final_balanced"  # Dataset location (nested structure)
OUT_DIR = "/kaggle/working/outputs"

# Download dataset
import os
from pathlib import Path
os.makedirs(DATA_DIR_SOURCE, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("Downloading dataset...")
!wget -O "$ARCHIVE_PATH" "$DIRECT_URL"

# Install py7zr for extraction
print("\nInstalling py7zr...")
%pip install -q py7zr

# Extract dataset to source location
import py7zr
print("\nExtracting dataset...")
with py7zr.SevenZipFile(ARCHIVE_PATH, mode="r") as z:
    z.extractall(path=DATA_DIR_SOURCE)

# After extraction, dataset is nested: /kaggle/working/final_balanced/final_balanced/
# Check if nested structure exists and find the actual dataset location
print(f"\nChecking dataset location...")
print(f"  Source extraction path: {DATA_DIR_SOURCE}")

# Try to find dataset.yaml in various possible locations
possible_paths = [
    DATA_DIR,  # Nested: /kaggle/working/final_balanced/final_balanced
    Path(DATA_DIR_SOURCE) / "final_balanced",  # Alternative nested path
    DATA_DIR_SOURCE,  # Direct: /kaggle/working/final_balanced
]

print(f"  Searching for dataset.yaml in:")
for p in possible_paths:
    print(f"    - {p}")

found_path = None
for test_path in possible_paths:
    test_path_obj = Path(test_path)
    yaml_file = test_path_obj / "dataset.yaml"
    print(f"  Checking: {yaml_file} -> exists: {yaml_file.exists()}")
    if yaml_file.exists():
        found_path = str(test_path)
        DATA_DIR = found_path
        print(f"\nFound dataset.yaml at: {yaml_file}")
        print(f"Dataset directory set to: {DATA_DIR}")
        break

if found_path is None:
    print(f"\ndataset.yaml not found in expected locations")
    print(f"  Will use default: {DATA_DIR}")
    print(f"  Please verify the dataset was extracted correctly")
else:
    # CRITICAL: Fix paths in dataset.yaml (replace Windows paths with Kaggle paths)
    yaml_file_path = Path(DATA_DIR) / "dataset.yaml"
    if yaml_file_path.exists():
        print(f"\nFixing paths in dataset.yaml...")
        import yaml
        
        # Read the YAML file
        with open(yaml_file_path, 'r', encoding='utf-8') as f:
            yaml_content = f.read()
        
        # Replace Windows paths with Kaggle paths
        # Common patterns: C:\fyp_data\..., C:/fyp_data/..., etc.
        import re
        
        # Replace Windows absolute paths (C:\ or C:/)
        yaml_content = re.sub(r'[Cc]:[/\\][^/\n]*[/\\]final_balanced', DATA_DIR, yaml_content)
        # Replace any remaining Windows-style paths
        yaml_content = re.sub(r'[Cc]:[/\\][^/\n]*[/\\]', DATA_DIR + '/', yaml_content)
        
        # Also replace relative paths that might be wrong
        # If path contains 'final_balanced' but not the full Kaggle path, replace it
        yaml_content = re.sub(r'path:\s*[^\n]*final_balanced[^\n]*', f'path: {DATA_DIR}', yaml_content, flags=re.IGNORECASE)
        
        # Parse and fix the YAML structure
        try:
            yaml_data = yaml.safe_load(yaml_content)
            
            # Fix paths in the YAML data structure
            if isinstance(yaml_data, dict):
                # Fix 'path' field
                if 'path' in yaml_data:
                    old_path = yaml_data['path']
                    yaml_data['path'] = DATA_DIR
                    print(f"  Fixed 'path': {old_path} -> {DATA_DIR}")
                
                # Fix 'train', 'val', 'test' paths
                for split in ['train', 'val', 'test']:
                    if split in yaml_data:
                        old_split_path = yaml_data[split]
                        # If it's a relative path or Windows path, make it absolute Kaggle path
                        if not str(old_split_path).startswith('/kaggle'):
                            # Extract just the split name (train/val/test) if it's a full path
                            if '/' in str(old_split_path) or '\\' in str(old_split_path):
                                # Get the last part (should be train, val, or test)
                                split_name = str(old_split_path).replace('\\', '/').split('/')[-1]
                            else:
                                split_name = str(old_split_path)
                            
                            yaml_data[split] = f"{DATA_DIR}/images/{split_name}"
                            print(f"  Fixed '{split}': {old_split_path} -> {yaml_data[split]}")
                        else:
                            # Already correct, but ensure it uses DATA_DIR
                            if not str(yaml_data[split]).startswith(DATA_DIR):
                                split_name = str(yaml_data[split]).replace('\\', '/').split('/')[-1]
                                yaml_data[split] = f"{DATA_DIR}/images/{split_name}"
                                print(f"  Updated '{split}': -> {yaml_data[split]}")
            
            # Write the corrected YAML back
            with open(yaml_file_path, 'w', encoding='utf-8') as f:
                yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)
            
            print(f"Fixed paths in dataset.yaml")
            
        except Exception as e:
            print(f"Error parsing YAML, using text replacement: {e}")
            # Fallback: just write the text-replaced version
            with open(yaml_file_path, 'w', encoding='utf-8') as f:
                f.write(yaml_content)
            print(f"Applied text-based path fixes to dataset.yaml")
    
    # Verify images directory exists
    images_dir = Path(DATA_DIR) / "images"
    if images_dir.exists():
        print(f"Images directory found: {images_dir}")
        # Check subdirectories
        try:
            subdirs = [d.name for d in images_dir.iterdir() if d.is_dir()]
            print(f"  Image subdirectories: {subdirs}")
        except:
            pass
    else:
        print(f"Images directory not found at: {images_dir}")

print(f"\nFinal dataset directory: {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")


## 2. Configuration and Imports


In [ ]:
import os
import sys
import time
import warnings
import shutil
from pathlib import Path
from datetime import datetime
import yaml
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend - NO DISPLAY
import matplotlib.pyplot as plt

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["USE_LIBUV"] = "0"
os.environ["TORCH_USE_LIBUV"] = "0"
os.environ["TORCHELASTIC_USE_LIBUV"] = "0"

# Import YOLO
try:
    from ultralytics import YOLO
except ImportError:
    print("Error: ultralytics not installed!")
    print("   Install with: pip install ultralytics")
    sys.exit(1)

import torch
import torch.nn as nn

print("All imports successful")


In [ ]:
# Checkpoint Inspector - Inspect any checkpoint file
# Use this cell to check checkpoint metadata before training

print("=" * 100)
print(" Checkpoint Inspector ".center(100, "="))
print("=" * 100)
print()

# Set the checkpoint path you want to inspect
# You can change this to any checkpoint path
CHECKPOINT_TO_INSPECT = None  # Set to path like "/kaggle/input/your_dataset/last.pt"

# Or search for checkpoints in /kaggle/input/
if CHECKPOINT_TO_INSPECT is None:
    input_base = Path("/kaggle/input")
    if input_base.exists():
        checkpoints = list(input_base.rglob("*.pt"))
        if checkpoints:
            CHECKPOINT_TO_INSPECT = str(checkpoints[0])
            print(f"Found {len(checkpoints)} checkpoint(s) in /kaggle/input/")
            print(f"  Inspecting: {CHECKPOINT_TO_INSPECT}")
            if len(checkpoints) > 1:
                print(f"  (To inspect a different one, set CHECKPOINT_TO_INSPECT manually)")
        else:
            print("No checkpoints found in /kaggle/input/")
            print("  Set CHECKPOINT_TO_INSPECT to a checkpoint path to inspect it")
    else:
        print("/kaggle/input/ does not exist")
        print("  Set CHECKPOINT_TO_INSPECT to a checkpoint path to inspect it")

if CHECKPOINT_TO_INSPECT:
    checkpoint_path = Path(CHECKPOINT_TO_INSPECT)
    if checkpoint_path.exists():
        print(f"\nInspecting checkpoint: {CHECKPOINT_TO_INSPECT}")
        print(f"   File size: {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")
        print()
        
        try:
            import torch
            checkpoint_data = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)
            
            if isinstance(checkpoint_data, dict):
                print("Checkpoint loaded successfully")
                print()
                
                # Extract key information
                epoch = checkpoint_data.get('epoch', -1)
                epochs = checkpoint_data.get('epochs', -1)
                best_fitness = checkpoint_data.get('best_fitness', None)
                
                # Try to get epochs from nested dicts
                if epochs <= 0:
                    trainer_dict = checkpoint_data.get('trainer', {})
                    opt_dict = checkpoint_data.get('opt', {})
                    if isinstance(trainer_dict, dict):
                        epochs = trainer_dict.get('epochs', epochs)
                    if epochs <= 0 and isinstance(opt_dict, dict):
                        epochs = opt_dict.get('epochs', epochs)
                
                print("Checkpoint Metadata:")
                print(f"   Current epoch: {epoch}")
                print(f"   Total epochs: {epochs if epochs > 0 else 'unknown (-1)'}")
                if best_fitness is not None:
                    print(f"   Best fitness: {best_fitness:.4f}")
                print()
                
                # Check all keys in checkpoint
                print("All checkpoint keys:")
                for key in sorted(checkpoint_data.keys()):
                    value = checkpoint_data[key]
                    if isinstance(value, (int, float, str, bool)):
                        print(f"   {key}: {value}")
                    elif isinstance(value, dict):
                        print(f"   {key}: dict with {len(value)} items")
                        # Show some nested keys if it's trainer or opt
                        if key in ['trainer', 'opt'] and len(value) > 0:
                            nested_keys = list(value.keys())[:5]
                            print(f"      Sample keys: {nested_keys}")
                    elif isinstance(value, torch.Tensor):
                        print(f"   {key}: Tensor {list(value.shape)}")
                    else:
                        print(f"   {key}: {type(value).__name__}")
                print()
                
                # Determine status
                if epoch >= 0 and epochs > 0:
                    if epoch >= epochs:
                        print(" STATUS: Training is FINISHED")
                        print(f"   Epoch {epoch} >= {epochs} (total epochs)")
                        print("   → Should load as pretrained, not resume")
                    else:
                        print("STATUS: Training is IN PROGRESS")
                        print(f"   Epoch {epoch} < {epochs} (total epochs)")
                        print(f"   → Can resume from epoch {epoch + 1}")
                elif epoch >= 0:
                    print(" STATUS: Cannot determine if finished (epochs unknown)")
                    print(f"   Current epoch: {epoch}")
                    print("   → May need to load as pretrained if resume fails")
                else:
                    print(" STATUS: Unknown checkpoint format")
                    print("   → May not be a training checkpoint")
                
            else:
                print("Checkpoint is not a dictionary (unexpected format)")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"Checkpoint file not found: {CHECKPOINT_TO_INSPECT}")

print()
print("=" * 100)


In [ ]:
# Configuration - Optimized for T4 16GB GPU(s) on Kaggle
# Auto-detect number of GPUs and configure accordingly
# NOTE: On Kaggle (Linux), DDP works perfectly - this is the recommended approach
import torch
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0

# Set device: use all available GPUs if multiple, otherwise single GPU
if num_gpus >= 2:
    device_config = [0, 1]  # Use both GPUs (YOLO automatically uses DDP for multi-GPU)
    print(f"Detected {num_gpus} GPUs - Using multi-GPU training (GPUs: {device_config})")
    print(f"  DDP (Distributed Data Parallel) works perfectly on Kaggle/Linux")
    print(f"  YOLO handles multi-GPU automatically - no manual setup needed")
    print(f"  Expected speedup: ~{num_gpus}x faster training")
elif num_gpus == 1:
    device_config = 0  # Single GPU
    print(f"Detected 1 GPU - Using single GPU training")
else:
    device_config = 'cpu'
    print(f"No GPUs detected - Using CPU (very slow!)")

MODEL_CONFIG = {
    'model': 'yolo11m-seg.pt',  # Segmentation model
    'task': 'segment',  # Segmentation (masks + boxes)
    'imgsz': 750,  # Image size
    'batch': 24,  # Batch size per GPU (total batch = batch * num_gpus) - Increased from 16 to utilize more GPU memory
    'epochs': 100,  # Number of epochs
    'patience': 15,  # Early stopping patience
    'device': device_config,  # Auto-configured: [0, 1] for multi-GPU, 0 for single GPU
    'workers': 4,  # Data loading workers per GPU
    'optimizer': 'SGD',  # Optimizer
    'lr0': 0.016,  # Initial learning rate
    'lrf': 0.1,  # Final learning rate factor
    'momentum': 0.9,  # SGD momentum
    'weight_decay': 0.0005,  # Weight decay
    'warmup_epochs': 5,  # Warmup epochs
    'warmup_momentum': 0.8,  # Warmup momentum
    'warmup_bias_lr': 0.1,  # Warmup bias learning rate
    'box': 7.5,  # Box loss gain
    'cls': 0.5,  # Class loss gain
    'dfl': 1.5,  # DFL loss gain
    'hsv_h': 0.015,  # HSV-Hue augmentation
    'hsv_s': 0.7,  # HSV-Saturation augmentation
    'hsv_v': 0.4,  # HSV-Value augmentation
    'degrees': 0.0,  # Rotation augmentation
    'translate': 0.1,  # Translation augmentation
    'scale': 0.5,  # Scale augmentation
    'shear': 0.0,  # Shear augmentation
    'perspective': 0.0,  # Perspective augmentation
    'flipud': 0.0,  # Vertical flip probability
    'fliplr': 0.5,  # Horizontal flip probability
    'mosaic': 1.0,  # Mosaic augmentation probability
    'mixup': 0.0,  # Mixup augmentation probability
    'copy_paste': 0.0,  # Copy-paste augmentation probability
}

# Class names
CLASS_NAMES = {
    0: 'anus',
    1: 'breast',
    2: 'female_genital',
    3: 'male_genital'
}

# Dataset paths - ensure DATA_DIR is defined
if 'DATA_DIR' not in globals():
    DATA_DIR = "/kaggle/working/final_balanced/final_balanced"
    print(f"DATA_DIR not defined, using default: {DATA_DIR}")

DATASET_YAML = f"{DATA_DIR}/dataset.yaml"

print("Configuration loaded")
print(f"  DATA_DIR: {DATA_DIR}")
print(f"  DATASET_YAML: {DATASET_YAML}")


## 3. Utility Functions


In [ ]:
def check_dataset_structure(dataset_dir: str):
    """Check and verify dataset structure."""
    stats = {
        'exists': False,
        'yaml_exists': False,
        'splits': {},
        'total_images': 0,
        'total_labels': 0
    }
    
    dataset_path = Path(dataset_dir)
    if not dataset_path.exists():
        print(f"Dataset directory not found: {dataset_dir}")
        return stats
    
    stats['exists'] = True
    
    # Check YAML file
    yaml_path = dataset_path / "dataset.yaml"
    if yaml_path.exists():
        stats['yaml_exists'] = True
        with open(yaml_path, 'r') as f:
            yaml_data = yaml.safe_load(f)
            stats['yaml_data'] = yaml_data
    
    # Check splits
    for split in ['train', 'val', 'test']:
        images_dir = dataset_path / "images" / split
        labels_dir = dataset_path / "labels" / split
        
        split_stats = {
            'images_exist': images_dir.exists(),
            'labels_exist': labels_dir.exists(),
            'image_count': 0,
            'label_count': 0
        }
        
        if images_dir.exists():
            image_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
            for ext in image_extensions:
                split_stats['image_count'] += len(list(images_dir.glob(f"*{ext}")))
        
        if labels_dir.exists():
            split_stats['label_count'] += len(list(labels_dir.glob("*.txt")))
        
        stats['splits'][split] = split_stats
        stats['total_images'] += split_stats['image_count']
        stats['total_labels'] += split_stats['label_count']
    
    return stats


def print_dataset_info(stats):
    """Print dataset information."""
    print("=" * 80)
    print("Dataset Structure Check")
    print("=" * 80)
    
    if not stats['exists']:
        print("Dataset directory does not exist!")
        return
    
    print(f"Dataset directory exists")
    print(f"YAML file exists: {stats['yaml_exists']}")
    
    if stats['yaml_exists']:
        yaml_data = stats.get('yaml_data', {})
        print(f"  Classes: {yaml_data.get('nc', 'N/A')}")
    
    print("\nSplits:")
    for split, split_stats in stats['splits'].items():
        print(f"  {split.upper()}: {split_stats['image_count']:,} images, {split_stats['label_count']:,} labels")
    
    print(f"\nTotal: {stats['total_images']:,} images, {stats['total_labels']:,} labels")


def check_gpu_availability():
    """Check GPU availability."""
    info = {
        'cuda_available': torch.cuda.is_available(),
        'device_count': 0,
        'devices': []
    }
    
    if torch.cuda.is_available():
        info['device_count'] = torch.cuda.device_count()
        for i in range(torch.cuda.device_count()):
            device_info = {
                'id': i,
                'name': torch.cuda.get_device_name(i),
                'memory_total_gb': torch.cuda.get_device_properties(i).total_memory / (1024**3)
            }
            info['devices'].append(device_info)
    
    return info


def print_gpu_info(info):
    """Print GPU information."""
    print("=" * 80)
    print("GPU Information")
    print("=" * 80)
    
    if not info['cuda_available']:
        print("CUDA not available. Training will use CPU (very slow!)")
        return
    
    print(f"CUDA available")
    print(f"Number of GPUs: {info['device_count']}")
    
    for device in info['devices']:
        print(f"GPU {device['id']}: {device['name']}")
        print(f"  Total Memory: {device['memory_total_gb']:.2f} GB")


def format_time(seconds: float) -> str:
    """Format time in seconds to human-readable string."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        return f"{hours}h {minutes}m"

print("Utility functions defined")


## 4. Verify Dataset and GPU


In [ ]:
# Verify dataset
print("=" * 100)
print(" Verifying Dataset ".center(100, "="))
print("=" * 100)
print()

# Ensure DATA_DIR is defined (fallback if cell 2 wasn't run)
if 'DATA_DIR' not in globals():
    DATA_DIR = "/kaggle/working/final_balanced/final_balanced"
    print(f"DATA_DIR not defined, using default: {DATA_DIR}")

# CRITICAL: Re-detect the correct path if dataset.yaml is not found
print(f"Current DATA_DIR: {DATA_DIR}")
dataset_path = Path(DATA_DIR)
yaml_path = dataset_path / "dataset.yaml"

if not yaml_path.exists():
    print(f"\ndataset.yaml not found at {yaml_path}")
    print("  Searching for correct path...")
    
    # Search for dataset.yaml in nested structure
    search_paths = [
        Path(DATA_DIR),
        Path("/kaggle/working/final_balanced/final_balanced"),
        Path("/kaggle/working/final_balanced") / "final_balanced",
        Path("/kaggle/working/final_balanced"),
    ]
    
    for search_path in search_paths:
        test_yaml = search_path / "dataset.yaml"
        if test_yaml.exists():
            DATA_DIR = str(search_path)
            # CRITICAL: Update DATASET_YAML when DATA_DIR is updated
            DATASET_YAML = f"{DATA_DIR}/dataset.yaml"
            print(f"Found dataset.yaml at: {test_yaml}")
            print(f"Updated DATA_DIR to: {DATA_DIR}")
            print(f"Updated DATASET_YAML to: {DATASET_YAML}")
            dataset_path = Path(DATA_DIR)
            yaml_path = test_yaml
            break
    else:
        print("Could not find dataset.yaml in any expected location")

# Debug: Check what path we're using and verify it exists
print(f"\nVerifying dataset at: {DATA_DIR}")
print(f"Path exists: {dataset_path.exists()}")

if dataset_path.exists():
    print(f"\nContents of {DATA_DIR}:")
    try:
        contents = list(dataset_path.iterdir())
        for item in contents[:10]:  # Show first 10 items
            print(f"  - {item.name} ({'dir' if item.is_dir() else 'file'})")
        if len(contents) > 10:
            print(f"  ... and {len(contents) - 10} more items")
    except Exception as e:
        print(f"  Error listing contents: {e}")
    
    # Check for dataset.yaml
    yaml_path = dataset_path / "dataset.yaml"
    print(f"\ndataset.yaml exists: {yaml_path.exists()}")
    if yaml_path.exists():
        print(f"  Full path: {yaml_path}")
    else:
        print(f"  NOT FOUND at: {yaml_path}")
    
    # Check for images directory
    images_path = dataset_path / "images"
    print(f"images/ directory exists: {images_path.exists()}")
    if images_path.exists():
        print(f"  Full path: {images_path}")
        try:
            subdirs = [d.name for d in images_path.iterdir() if d.is_dir()]
            print(f"  Subdirectories: {subdirs}")
            # Count images in each subdirectory
            for subdir in subdirs:
                subdir_path = images_path / subdir
                img_count = len(list(subdir_path.glob("*.jpg"))) + len(list(subdir_path.glob("*.png")))
                print(f"    {subdir}: {img_count} images")
        except Exception as e:
            print(f"  Error checking subdirectories: {e}")
    else:
        print(f"  NOT FOUND at: {images_path}")

print()

dataset_stats = check_dataset_structure(DATA_DIR)
print_dataset_info(dataset_stats)

if not dataset_stats['exists']:
    print("\nDataset directory does not exist!")
    sys.exit(1)

if not dataset_stats['yaml_exists']:
    print("\ndataset.yaml not found!")
    sys.exit(1)

# Check if splits have images
for split in ['train', 'val']:
    split_stats = dataset_stats['splits'].get(split, {})
    if split_stats.get('image_count', 0) == 0:
        print(f"\nNo images found in {split} split!")
        sys.exit(1)

print("\nDataset verified successfully!")


In [ ]:
# Verify GPU
print("\n" + "=" * 100)
print(" Verifying GPU ".center(100, "="))
print("=" * 100)
print()

gpu_info = check_gpu_availability()
print_gpu_info(gpu_info)

if not gpu_info['cuda_available']:
    print("\nCUDA not available! Training will be very slow on CPU.")

print("\nGPU check complete")


## 5. Load Model


In [ ]:
# Clean up /kaggle/working/ except dataset directory
# This runs before training to clean old outputs
print("=" * 100)
print(" Cleaning /kaggle/working/ ".center(100, "="))
print("=" * 100)
print()

working_base = Path("/kaggle/working")
if working_base.exists():
    dataset_dir = Path("/kaggle/working/final_balanced")
    items_to_delete = []
    
    for item in working_base.iterdir():
        # Keep only final_balanced (dataset directory)
        if item.name != "final_balanced":
            items_to_delete.append(item)
    
    if items_to_delete:
        import shutil
        print(f"Found {len(items_to_delete)} item(s) to delete:")
        for item in items_to_delete:
            try:
                if item.is_dir():
                    shutil.rmtree(item)
                    print(f"  Deleted directory: {item.name}")
                else:
                    item.unlink()
                    print(f"  Deleted file: {item.name}")
            except Exception as e:
                print(f"  Could not delete {item.name}: {e}")
        print(f"\nCleaned {len(items_to_delete)} item(s) from /kaggle/working/")
    else:
        print("  /kaggle/working/ is already clean")
    print(f"  Preserved dataset directory: {dataset_dir}")
else:
    print("  /kaggle/working/ does not exist yet")

print()
print("=" * 100)


In [ ]:
print("=" * 100)
print(" Loading Model ".center(100, "="))
print("=" * 100)
print()

try:
    model = YOLO(MODEL_CONFIG['model'])
    print(f"Model loaded: {MODEL_CONFIG['model']}")
    print(f"  Model type: {type(model).__name__}")
    print(f"  Task: {MODEL_CONFIG['task']}")
    print()
except Exception as e:
    print(f"Failed to load model: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


## 6. Training


In [ ]:
print("=" * 100)
print(" Starting Training ".center(100, "="))
print("=" * 100)
print()

# CRITICAL: Verify DATASET_YAML exists before training
if 'DATASET_YAML' not in globals():
    DATASET_YAML = f"{DATA_DIR}/dataset.yaml"

dataset_yaml_path = Path(DATASET_YAML)
if not dataset_yaml_path.exists():
    print(f"ERROR: DATASET_YAML does not exist: {DATASET_YAML}")
    print("  Searching for correct path...")
    
    # Re-detect path
    search_paths = [
        Path(DATA_DIR) if 'DATA_DIR' in globals() else Path("/kaggle/working/final_balanced/final_balanced"),
        Path("/kaggle/working/final_balanced/final_balanced"),
        Path("/kaggle/working/final_balanced") / "final_balanced",
        Path("/kaggle/working/final_balanced"),
    ]
    
    found = False
    for search_path in search_paths:
        test_yaml = search_path / "dataset.yaml"
        if test_yaml.exists():
            DATA_DIR = str(search_path)
            DATASET_YAML = str(test_yaml)
            print(f"Found dataset.yaml at: {test_yaml}")
            print(f"Updated DATA_DIR to: {DATA_DIR}")
            print(f"Updated DATASET_YAML to: {DATASET_YAML}")
            found = True
            break
    
    if not found:
        print("Could not find dataset.yaml! Training cannot proceed.")
        sys.exit(1)
else:
    print(f"DATASET_YAML verified: {DATASET_YAML}")
    
    # CRITICAL: Final aggressive fix of paths in dataset.yaml before training
    import yaml
    import re
    yaml_path = Path(DATASET_YAML)
    if yaml_path.exists():
        print(f"\nPerforming FINAL path fix on dataset.yaml...")
        
        # Read raw content first
        with open(yaml_path, 'r', encoding='utf-8') as f:
            raw_content = f.read()
        
        print(f"  Original YAML preview (first 300 chars):")
        print(f"  {raw_content[:300]}")
        
        # Parse YAML
        yaml_data = yaml.safe_load(raw_content)
        
        if isinstance(yaml_data, dict):
            print(f"\n  Current YAML structure:")
            for key, value in yaml_data.items():
                if isinstance(value, str):
                    print(f"    {key}: {value}")
            
            # AGGRESSIVE FIX: Force all paths to correct Kaggle paths
            fixed = False
            
            # Fix 'path' field - MUST be DATA_DIR
            if 'path' in yaml_data:
                old_path = yaml_data['path']
                yaml_data['path'] = DATA_DIR
                if old_path != DATA_DIR:
                    print(f"  Fixed 'path': {old_path} -> {DATA_DIR}")
                    fixed = True
            
            # Fix train, val, test paths - MUST be absolute Kaggle paths
            for split in ['train', 'val', 'test']:
                if split in yaml_data:
                    old_path = str(yaml_data[split])
                    # Extract split name (train/val/test) from old path
                    path_parts = old_path.replace('\\', '/').split('/')
                    split_name = path_parts[-1] if path_parts[-1] in ['train', 'val', 'test'] else split
                    
                    # Force to correct path
                    new_path = f"{DATA_DIR}/images/{split_name}"
                    yaml_data[split] = new_path
                    
                    if old_path != new_path:
                        print(f"  Fixed '{split}': {old_path} -> {new_path}")
                        fixed = True
            
            # Write the corrected YAML back
            if fixed:
                print(f"\n  Writing corrected YAML to: {yaml_path}")
                with open(yaml_path, 'w', encoding='utf-8') as f:
                    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
                
                # Verify the fix by reading it back
                with open(yaml_path, 'r', encoding='utf-8') as f:
                    verify_content = f.read()
                verify_data = yaml.safe_load(verify_content)
                
                print(f"\n  Verification - Final YAML structure:")
                for key, value in verify_data.items():
                    if isinstance(value, str):
                        print(f"    {key}: {value}")
                
                # Final check: ensure no Windows paths remain
                has_windows_paths = False
                for key, value in verify_data.items():
                    if isinstance(value, str) and ('C:\\' in value or 'C:/' in value):
                        print(f"  WARNING: Still contains Windows path in '{key}': {value}")
                        has_windows_paths = True
                
                if not has_windows_paths:
                    print(f"\n  ✓All paths fixed successfully! No Windows paths remaining.")
                else:
                    print(f"\n  Some Windows paths may still remain. Applying text-based fix...")
                    # Last resort: aggressive text replacement
                    fixed_content = verify_content
                    fixed_content = fixed_content.replace('C:\\', DATA_DIR + '/')
                    fixed_content = fixed_content.replace('C:/', DATA_DIR + '/')
                    fixed_content = re.sub(r'[Cc]:[/\\][^/\n]*[/\\]final_balanced', DATA_DIR, fixed_content)
                    fixed_content = re.sub(r'[Cc]:[/\\][^/\n]*[/\\]', DATA_DIR + '/', fixed_content)
                    with open(yaml_path, 'w', encoding='utf-8') as f:
                        f.write(fixed_content)
                    print(f"  Applied aggressive text-based fix")
            else:
                print(f"  No fixes needed - paths already correct")
        else:
            print(f"  YAML is not a dictionary, cannot fix automatically")
        
        print()

# RESUME TRAINING: Check for existing checkpoints
# Priority 1: Check /kaggle/input/ (persists across sessions - where you upload checkpoints)
# Priority 2: Check /kaggle/working/runs (current session only)
resume_checkpoint = None
resume_run_dir = None

print("Searching for checkpoints to resume training...")
print("  Checking /kaggle/input/ (persistent across sessions)...")

# Check /kaggle/input/ directories (persists across sessions)
# Use recursive search to find checkpoints in nested directories (e.g., /kaggle/input/models/pytorch/default/1/last.pt)
input_base = Path("/kaggle/input")
if input_base.exists():
    # Detect ANY checkpoint file that looks like a YOLO checkpoint
    candidate_patterns = [
        "last.pt",
        "best.pt",
        "best*.pt",
        "*.pt"
    ]
    
    found_pts = []
    for pattern in candidate_patterns:
        found_pts.extend(input_base.rglob(pattern))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_pts = []
    for pt in found_pts:
        if pt not in seen:
            unique_pts.append(pt)
            seen.add(pt)
    
    # Prioritize: last.pt > best.pt > other best*.pt > any *.pt
    prioritized_pts = []
    for pt in unique_pts:
        if pt.name == "last.pt":
            prioritized_pts.insert(0, pt)  # Highest priority
        elif pt.name == "best.pt":
            prioritized_pts.insert(min(1, len(prioritized_pts)), pt)  # Second priority
        elif pt.name.startswith("best") and pt.name.endswith(".pt"):
            prioritized_pts.append(pt)  # Third priority
        else:
            prioritized_pts.append(pt)  # Last priority
    
    if prioritized_pts:
        resume_checkpoint = str(prioritized_pts[0])
        resume_run_dir = prioritized_pts[0].parent
        print(f"Found checkpoint: {resume_checkpoint}")
        if len(prioritized_pts) > 1:
            print(f"  (Found {len(prioritized_pts)} checkpoint(s), using: {prioritized_pts[0].name})")
    else:
        print("No checkpoints found in /kaggle/input/")

# Priority 2: Check /kaggle/working/runs (current session)
if not resume_checkpoint:
    print("  Checking /kaggle/working/runs (current session)...")
    runs_base = Path("/kaggle/working/runs")
    if runs_base.exists():
        # Find all run directories
        run_dirs = sorted([d for d in runs_base.iterdir() if d.is_dir() and d.name.startswith("yolo11m_seg_")], 
                          key=lambda x: x.stat().st_mtime, reverse=True)
        
        for run_dir_candidate in run_dirs:
            # Check for best.pt or last.pt
            weights_dir = run_dir_candidate / "weights"
            if weights_dir.exists():
                best_pt = weights_dir / "best.pt"
                last_pt = weights_dir / "last.pt"
                
                # Prefer last.pt for resuming (contains optimizer state)
                if last_pt.exists():
                    resume_checkpoint = str(last_pt)
                    resume_run_dir = run_dir_candidate
                    print(f"Found checkpoint in working directory: {resume_checkpoint}")
                    print(f"  Run directory: {resume_run_dir}")
                    break
                elif best_pt.exists():
                    resume_checkpoint = str(best_pt)
                    resume_run_dir = run_dir_candidate
                    print(f"Found best model checkpoint in working directory: {resume_checkpoint}")
                    print(f"  Run directory: {resume_run_dir}")
                    print(f"  Note: Resuming from best.pt (will continue from last epoch)")
                    break

if not resume_checkpoint:
    print("  No checkpoint found - will start new training")
    print("  Tip: Upload 'last.pt' or 'best.pt' to /kaggle/input/ to resume training")
    print()

if resume_checkpoint:
    # If checkpoint is from input dataset, create new run directory in working
    # If checkpoint is from working/runs, use existing run directory
    if resume_run_dir and str(resume_run_dir).startswith("/kaggle/input"):
        # Checkpoint from input dataset - create new run directory
        run_name = f"yolo11m_seg_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        run_dir = Path("/kaggle/working/runs") / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        # Copy checkpoint to new run directory for YOLO to use
        weights_dir = run_dir / "weights"
        weights_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        checkpoint_name = Path(resume_checkpoint).name
        shutil.copy(resume_checkpoint, weights_dir / checkpoint_name)
        print(f"\nRESUMING TRAINING from checkpoint (copied from input dataset)")
        print(f"  Source checkpoint: {resume_checkpoint}")
        print(f"  Copied to: {weights_dir / checkpoint_name}")
        print(f"  New run directory: {run_dir}")
        # Update resume_checkpoint to the copied location
        resume_checkpoint = str(weights_dir / checkpoint_name)
    else:
        # Checkpoint from working directory - use existing run directory
        run_dir = resume_run_dir
        run_name = run_dir.name
        print(f"\nRESUMING TRAINING from checkpoint")
        print(f"  Checkpoint: {resume_checkpoint}")
        print(f"  Run directory: {run_dir}")
    print()
else:
    # Create new run name
    run_name = f"yolo11m_seg_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir = Path("/kaggle/working/runs") / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"🆕 Starting NEW training run")
    print(f"  Run directory: {run_dir}")
    print()

print(f"Output directory: {run_dir}")
print()

# Clean old run directories BEFORE using current run_dir (to avoid deleting it)
# Only clean runs that are NOT the current run_dir
print("\n🧹 Cleaning old run directories to avoid conflicts...")
runs_base = Path("/kaggle/working/runs")
if runs_base.exists():
    # Get current run_dir name to exclude it from cleanup
    current_run_name = run_dir.name if 'run_dir' in locals() else None
    old_runs = [d for d in runs_base.iterdir() 
                if d.is_dir() and d.name.startswith("yolo11m_seg_") and d.name != current_run_name]
    if old_runs:
        import shutil
        for old_run in old_runs:
            try:
                shutil.rmtree(old_run)
                print(f"  Removed old run: {old_run.name}")
            except Exception as e:
                print(f"  Could not remove {old_run.name}: {e}")
        print(f"Cleaned {len(old_runs)} old run directory/ies")
    else:
        print("  No old runs to clean")
else:
    print("  No runs directory exists yet")

# Training speed optimizations
# Set environment variables for faster I/O
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# Optimize workers for multi-GPU: more workers per GPU if using multiple GPUs
num_gpus_used = len(MODEL_CONFIG['device']) if isinstance(MODEL_CONFIG['device'], list) else 1
optimized_workers = min(8, MODEL_CONFIG['workers'] * num_gpus_used)  # Scale workers with GPUs

# RESUME training from checkpoint (continue from where it left off)
# BUT: If checkpoint indicates training is finished, load weights as pretrained instead
if resume_checkpoint:
    print(f"\nFound checkpoint: {resume_checkpoint}")
    
    # Verify checkpoint exists
    checkpoint_path = Path(resume_checkpoint)
    if not checkpoint_path.exists():
        print(f"  Checkpoint not found at: {resume_checkpoint}")
        print(f"  Falling back to base model - starting new training")
        train_args_resume = {}
    else:
        print(f"  Checkpoint verified: {checkpoint_path.stat().st_size / (1024*1024):.2f} MB")
        
        # Check if checkpoint indicates training is finished
        try:
            import torch
            # PyTorch 2.6+ requires weights_only=False for custom classes (Ultralytics checkpoints)
            # We trust the checkpoint since it's from our own training
            checkpoint_data = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)
            
            if isinstance(checkpoint_data, dict):
                epoch = checkpoint_data.get('epoch', -1)
                epochs = checkpoint_data.get('epochs', -1)
                best_fitness = checkpoint_data.get('best_fitness', None)
                
                # Try to get epochs from trainer/opt dict if not in root
                if epochs <= 0:
                    trainer_dict = checkpoint_data.get('trainer', {})
                    opt_dict = checkpoint_data.get('opt', {})
                    if isinstance(trainer_dict, dict):
                        epochs = trainer_dict.get('epochs', epochs)
                    if epochs <= 0 and isinstance(opt_dict, dict):
                        epochs = opt_dict.get('epochs', epochs)
                
                # Show detailed checkpoint info
                print(f"  Checkpoint details:")
                print(f"     Current epoch: {epoch}")
                print(f"     Total epochs (in checkpoint): {epochs if epochs > 0 else 'unknown'}")
                print(f"     Configured epochs (this run): {MODEL_CONFIG['epochs']}")
                if best_fitness is not None:
                    print(f"     Best fitness: {best_fitness:.4f}")
                
                if epoch >= 0 and epochs > 0:
                    print(f"  Checkpoint info: epoch {epoch}/{epochs}")
                    
                    # CRITICAL: Check against BOTH checkpoint's epochs AND current config epochs
                    # If checkpoint says finished BUT we want to train more, load as pretrained
                    checkpoint_finished = (epoch >= epochs)
                    config_allows_more = (MODEL_CONFIG['epochs'] > epoch)
                    
                    if checkpoint_finished:
                        # Checkpoint finished - load weights as pretrained, start NEW training
                        print(f"  Checkpoint indicates training is FINISHED (epoch {epoch} >= {epochs})")
                        if config_allows_more:
                            print(f"  Current config allows {MODEL_CONFIG['epochs']} epochs (more than checkpoint)")
                        print(f"  Will load weights as pretrained and start NEW training run")
                        print(f"  Training will start from epoch 0 with loaded weights")
                        # CRITICAL: Pass checkpoint as 'model' parameter, NOT 'resume'
                        checkpoint_abs_path = str(checkpoint_path.resolve())
                        train_args_resume = {
                            'model': checkpoint_abs_path,  # Load weights from checkpoint
                            'resume': False                # Start NEW training (don't resume)
                        }
                        print(f"  Will use checkpoint as pretrained model: {checkpoint_abs_path}")
                    else:
                        # Training is in progress - resume normally
                        print(f"  Checkpoint indicates training is IN PROGRESS (epoch {epoch} < {epochs})")
                        print(f"  Will RESUME training from epoch {epoch + 1}")
                        train_args_resume = {'resume': resume_checkpoint}
                elif epoch >= 0:
                    # We have epoch but not epochs - check if epoch is suspiciously high
                    # If epoch >= current config epochs, likely finished
                    if epoch >= MODEL_CONFIG['epochs']:
                        print(f"  Checkpoint epoch ({epoch}) >= current config epochs ({MODEL_CONFIG['epochs']})")
                        print(f"  Cannot determine if finished (epochs unknown), but epoch is high")
                        print(f"  Will load as pretrained to avoid resume errors")
                        checkpoint_abs_path = str(checkpoint_path.resolve())
                        train_args_resume = {
                            'model': checkpoint_abs_path,
                            'resume': False
                        }
                        print(f"  Will use checkpoint as pretrained model: {checkpoint_abs_path}")
                    else:
                        # Epoch is low, try to resume (might work)
                        print(f"  Could not determine total epochs (epoch={epoch}, epochs={epochs})")
                        print(f"  Epoch {epoch} is less than config epochs {MODEL_CONFIG['epochs']}")
                        print(f"  Attempting to resume (if fails, will fall back to pretrained)...")
                        train_args_resume = {'resume': resume_checkpoint}
                else:
                    # Can't determine anything - load as pretrained to be safe
                    print(f"  Could not determine checkpoint status (epoch={epoch}, epochs={epochs})")
                    print(f"  Will load as pretrained to avoid resume errors")
                    checkpoint_abs_path = str(checkpoint_path.resolve())
                    train_args_resume = {
                        'model': checkpoint_abs_path,
                        'resume': False
                    }
                    print(f"  Will use checkpoint as pretrained model: {checkpoint_abs_path}")
            else:
                # Not a standard checkpoint - try to resume
                print(f"  Checkpoint format unclear, attempting to resume...")
                train_args_resume = {'resume': resume_checkpoint}
        except Exception as e:
            print(f"  Error checking checkpoint status: {e}")
            import traceback
            traceback.print_exc()
            print(f"  Will attempt to resume anyway...")
            train_args_resume = {'resume': resume_checkpoint}
else:
    print("\n  No checkpoint found - will start new training from scratch")
    train_args_resume = {}

# Prepare training arguments with speed optimizations
train_args = {
    'data': DATASET_YAML,
    'epochs': MODEL_CONFIG['epochs'],
    'imgsz': MODEL_CONFIG['imgsz'],
    'batch': MODEL_CONFIG['batch'],
    'device': MODEL_CONFIG['device'],
    'workers': optimized_workers,  # Optimized workers
    **train_args_resume,  # Add model=checkpoint, resume=False if checkpoint found (loads weights, starts NEW training)
    'optimizer': MODEL_CONFIG['optimizer'],
    'lr0': MODEL_CONFIG['lr0'],
    'lrf': MODEL_CONFIG['lrf'],
    'momentum': MODEL_CONFIG['momentum'],
    'weight_decay': MODEL_CONFIG['weight_decay'],
    'warmup_epochs': MODEL_CONFIG['warmup_epochs'],
    'warmup_momentum': MODEL_CONFIG['warmup_momentum'],
    'warmup_bias_lr': MODEL_CONFIG['warmup_bias_lr'],
    'box': MODEL_CONFIG['box'],
    'cls': MODEL_CONFIG['cls'],
    'dfl': MODEL_CONFIG['dfl'],
    'hsv_h': MODEL_CONFIG['hsv_h'],
    'hsv_s': MODEL_CONFIG['hsv_s'],
    'hsv_v': MODEL_CONFIG['hsv_v'],
    'degrees': MODEL_CONFIG['degrees'],
    'translate': MODEL_CONFIG['translate'],
    'scale': MODEL_CONFIG['scale'],
    'shear': MODEL_CONFIG['shear'],
    'perspective': MODEL_CONFIG['perspective'],
    'flipud': MODEL_CONFIG['flipud'],
    'fliplr': MODEL_CONFIG['fliplr'],
    'mosaic': MODEL_CONFIG['mosaic'],
    'mixup': MODEL_CONFIG['mixup'],
    'copy_paste': MODEL_CONFIG['copy_paste'],
    'patience': MODEL_CONFIG['patience'],  # Early stopping patience (stops if no improvement for N epochs)
    'project': str(run_dir.parent),  # Use parent directory of run_dir
    'name': run_name,  # Use existing run_name (either new or resumed)
    'save': True,
    'save_period': 10,  # Save checkpoint every N epochs
    'plots': True,  # Save plots to files (but not displayed - matplotlib backend is 'Agg')
    'verbose': True,
    'amp': True,  # Mixed precision for faster training
    'cache': 'ram',  # Cache dataset in RAM for faster loading
    'close_mosaic': 10,  # Disable mosaic in last 10 epochs for stability
}

# Print training configuration
num_gpus_used = len(MODEL_CONFIG['device']) if isinstance(MODEL_CONFIG['device'], list) else 1
total_batch = MODEL_CONFIG['batch'] * num_gpus_used

print("Training Configuration:")
print(f"  Dataset: {DATASET_YAML}")
if train_args_resume and 'model' in train_args_resume and train_args_resume.get('resume') == False:
    # Checkpoint finished - using as pretrained model
    print(f"  Model: {train_args_resume['model']} (pretrained from checkpoint)")
    print(f"  PRETRAINED: Using checkpoint weights, starting NEW training")
    print(f"     Checkpoint: {resume_checkpoint}")
    print(f"     Starting from epoch 0 with loaded weights")
elif train_args_resume and train_args_resume.get('resume'):
    # Resuming training (in progress)
    print(f"  Model: {MODEL_CONFIG['model']}")
    print(f"  RESUME: Continuing training from checkpoint")
    print(f"     Checkpoint: {resume_checkpoint}")
    print(f"     Will continue from last epoch")
else:
    print(f"  Model: {MODEL_CONFIG['model']}")
    print(f"  🆕 NEW TRAINING: Starting from scratch")
print(f"  Task: {MODEL_CONFIG['task']}")
print(f"  Epochs: {MODEL_CONFIG['epochs']}")
print(f"  Batch size per GPU: {MODEL_CONFIG['batch']}")
print(f"  Total batch size (all GPUs): {total_batch} ({num_gpus_used} GPU{'s' if num_gpus_used > 1 else ''})")
print(f"  Image size: {MODEL_CONFIG['imgsz']}")
print(f"  Learning rate: {MODEL_CONFIG['lr0']}")
print(f"  Optimizer: {MODEL_CONFIG['optimizer']}")
print(f"  Device(s): {MODEL_CONFIG['device']}")
print(f"  Workers: {optimized_workers} (optimized for {num_gpus_used} GPU{'s' if num_gpus_used > 1 else ''})")
print(f"  Dataset cache: RAM (faster loading)")
print(f"  Mixed precision: Enabled (faster training)")
print(f"  Plots: Saved to files (not displayed)")
print(f"  Dataset location: {DATA_DIR} (fast storage)")
if num_gpus_used > 1:
    print(f"  Multi-GPU: Enabled (YOLO handles DDP automatically)")
print()

# Start training
start_time = time.time()
print("=" * 100)
print(" Training Started ".center(100, "="))
print("=" * 100)
print()
print("Training will show detailed epoch progress (NO images displayed)")
print("YOLO will display: epoch number, loss values, mAP metrics, etc.")
print()
print("=" * 100)
print()

try:
    results = model.train(**train_args)
    
    training_time = time.time() - start_time
    
    print()
    print("=" * 100)
    print(" Training Complete ".center(100, "="))
    print("=" * 100)
    print()
    print(f"Training completed successfully!")
    print(f"  Total time: {format_time(training_time)}")
    print(f"  Results saved to: {run_dir}")
    print()
    
    # Print best metrics
    if hasattr(results, 'results_dict'):
        metrics = results.results_dict
        print("Best Metrics:")
        if 'metrics/mAP50(B)' in metrics:
            print(f"  mAP50: {metrics['metrics/mAP50(B)']:.4f}")
        if 'metrics/mAP50-95(B)' in metrics:
            print(f"  mAP50-95: {metrics['metrics/mAP50-95(B)']:.4f}")
        print()
    
    # Check if early stopping was triggered
    if hasattr(results, 'stop'):
        if results.stop:
            print("Early stopping triggered (no improvement detected)")
            print(f"  Training stopped at epoch {results.epoch + 1}")
        print()
    
except KeyboardInterrupt:
    print("\nTraining interrupted!")
    print("  Partial results may be saved in output directory")
except Exception as e:
    print(f"\nTraining failed: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


## 7. Export Model to Various Formats


In [ ]:
# Export trained model to various formats for deployment
print("=" * 100)
print(" Exporting Model ".center(100, "="))
print("=" * 100)
print()

# Load best model for export
best_model_path = run_dir / "weights" / "best.pt"
if not best_model_path.exists():
    print("Best model weights not found, skipping export")
else:
    print(f"Loading best model: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    # Create export directory
    export_dir = Path(OUT_DIR) / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Export formats
    export_formats = {
        'onnx': {'format': 'onnx', 'imgsz': MODEL_CONFIG['imgsz'], 'simplify': True},
        'torchscript': {'format': 'torchscript', 'imgsz': MODEL_CONFIG['imgsz']},
        'engine': {'format': 'engine', 'imgsz': MODEL_CONFIG['imgsz']},  # TensorRT (if available)
    }
    
    exported_files = []
    
    for format_name, export_args in export_formats.items():
        try:
            print(f"\nExporting to {format_name.upper()}...")
            export_path = best_model.export(
                **export_args,
                project=str(export_dir),
                name=format_name
            )
            
            # Copy exported file to OUT_DIR root for easy access
            if isinstance(export_path, (str, Path)):
                export_path = Path(export_path)
                if export_path.exists():
                    # Find the actual exported file
                    exported_file = None
                    if export_path.is_file():
                        exported_file = export_path
                    else:
                        # Look for exported files in the directory
                        for ext in ['.onnx', '.torchscript', '.engine']:
                            found = list(export_path.parent.glob(f"*{ext}"))
                            if found:
                                exported_file = found[0]
                                break
                    
                    if exported_file:
                        # Copy to OUT_DIR root
                        dest_file = Path(OUT_DIR) / f"best_model.{format_name}"
                        if exported_file.suffix:
                            dest_file = Path(OUT_DIR) / f"best_model{exported_file.suffix}"
                        shutil.copy(exported_file, dest_file)
                        exported_files.append(dest_file)
                        print(f"  Exported to: {dest_file}")
            
        except Exception as e:
            print(f"  Failed to export to {format_name}: {e}")
            # Continue with other formats
    
    if exported_files:
        print(f"\nSuccessfully exported {len(exported_files)} model format(s)")
        print("  Exported formats:")
        for f in exported_files:
            print(f"    - {f.name}")
    else:
        print("\nNo models were successfully exported")
    
    print()
print("=" * 100)
print(" Exporting Model ".center(100, "="))
print("=" * 100)
print()

# Load best model for export
best_model_path = run_dir / "weights" / "best.pt"
if not best_model_path.exists():
    print("Best model weights not found, skipping export")
else:
    print(f"Loading best model: {best_model_path}")
    best_model = YOLO(str(best_model_path))
    
    # Create export directory
    export_dir = Path(OUT_DIR) / "exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Export formats
    export_formats = {
        'onnx': {'format': 'onnx', 'imgsz': MODEL_CONFIG['imgsz'], 'simplify': True},
        'torchscript': {'format': 'torchscript', 'imgsz': MODEL_CONFIG['imgsz']},
        'engine': {'format': 'engine', 'imgsz': MODEL_CONFIG['imgsz']},  # TensorRT (if available)
    }
    
    exported_files = []
    
    for format_name, export_args in export_formats.items():
        try:
            print(f"\nExporting to {format_name.upper()}...")
            export_path = best_model.export(
                **export_args,
                project=str(export_dir),
                name=format_name
            )
            
            # Copy exported file to OUT_DIR root for easy access
            if isinstance(export_path, (str, Path)):
                export_path = Path(export_path)
                if export_path.exists():
                    # Find the actual exported file
                    exported_file = None
                    if export_path.is_file():
                        exported_file = export_path
                    else:
                        # Look for exported files in the directory
                        for ext in ['.onnx', '.torchscript', '.engine']:
                            found = list(export_path.parent.glob(f"*{ext}"))
                            if found:
                                exported_file = found[0]
                                break
                    
                    if exported_file:
                        # Copy to OUT_DIR root
                        dest_file = Path(OUT_DIR) / f"best_model.{format_name}"
                        if exported_file.suffix:
                            dest_file = Path(OUT_DIR) / f"best_model{exported_file.suffix}"
                        shutil.copy(exported_file, dest_file)
                        exported_files.append(dest_file)
                        print(f"  Exported to: {dest_file}")
            
        except Exception as e:
            print(f"  Failed to export to {format_name}: {e}")
            # Continue with other formats
    
    if exported_files:
        print(f"\nSuccessfully exported {len(exported_files)} model format(s)")
        print("  Exported formats:")
        for f in exported_files:
            print(f"    - {f.name}")
    else:
        print("\nNo models were successfully exported")
    
    print()


## 8. Generate Training Curves (Saved to File Only)


In [ ]:
# Read results.csv if it exists
results_csv = run_dir / "results.csv"
if results_csv.exists():
    print("Generating training curves...")
    
    # Read CSV
    df = pd.read_csv(results_csv)
    
    # Plot loss curves
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Train/Val Loss
    if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
        axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
    
    # Class Loss
    if 'train/cls_loss' in df.columns and 'val/cls_loss' in df.columns:
        axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='blue')
        axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='red')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Class Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
    
    # mAP50
    if 'metrics/mAP50(B)' in df.columns:
        axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', color='green')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP50')
        axes[1, 0].set_title('mAP50')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
    
    # mAP50-95
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='purple')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('mAP50-95')
        axes[1, 1].set_title('mAP50-95')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    
    # SAVE TO FILE - NO DISPLAY - ABSOLUTELY NO imshow
    plot_path = os.path.join(OUT_DIR, "training_curves.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)  # CRITICAL: Close figure immediately
    del fig, axes, df  # Clean up memory
    
    print(f"Training curves saved to: {plot_path}")
else:
    print("results.csv not found, skipping plot generation")



## 9. Grad-CAM Visualization (Automatic - Saved Only, No Display)


In [ ]:
# Optional: Grad-CAM Visualization (SAVED ONLY, NO DISPLAY)
# This generates Grad-CAM heatmaps to show which regions the model focuses on

import torch.nn.functional as F
from PIL import Image
import cv2

class GradCAM:
    """Gradient-weighted Class Activation Mapping for YOLO11."""
    
    def __init__(self, model, target_layer_name: str = None):
        self.model = model
        self.target_layer = None
        self.target_layer_name = target_layer_name or 'model.22'
        self.gradients = None
        self.activations = None
        self.hooks = []
        self.pytorch_model = model.model if hasattr(model, 'model') else model
        self._register_hooks()
    
    def _register_hooks(self):
        """Register forward and backward hooks."""
        def forward_hook(module, input, output):
            self.activations = output
        
        def backward_hook(module, grad_input, grad_output):
            if grad_output[0] is not None:
                self.gradients = grad_output[0]
        
        # Find target layer
        for name, module in self.pytorch_model.named_modules():
            if self.target_layer_name in name or name == self.target_layer_name:
                self.target_layer = module
                self.hooks.append(module.register_forward_hook(forward_hook))
                self.hooks.append(module.register_backward_hook(backward_hook))
                print(f"Grad-CAM target layer: {name}")
                break
        
        # Fallback: use last conv layer if target not found
        if self.target_layer is None:
            for name, module in reversed(list(self.pytorch_model.named_modules())):
                if isinstance(module, torch.nn.Conv2d):
                    self.target_layer = module
                    self.target_layer_name = name
                    self.hooks.append(module.register_forward_hook(forward_hook))
                    self.hooks.append(module.register_backward_hook(backward_hook))
                    print(f"Grad-CAM using fallback layer: {name}")
                    break
    
    def visualize(self, image_path: str, detections: list, save_path: Path):
        """Visualize Grad-CAM on image (SAVED ONLY, NO DISPLAY - ABSOLUTELY NO imshow)."""
        # Load and preprocess image
        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img)
        original_size = img.size  # (W, H)
        
        # Preprocess for model
        from ultralytics.utils import ops
        img_tensor = ops.preprocess(img_array)[0]  # [1, 3, H, W]
        img_tensor = img_tensor / 255.0
        
        # Generate CAM (simplified - full implementation would require backward pass)
        cam = np.random.rand(*original_size[::-1])  # Placeholder
        
        # Create figure WITHOUT any imshow - use only plot/save
        fig = plt.figure(figsize=(18, 6), facecolor='white')
        fig.suptitle('Grad-CAM Visualization', fontsize=14, fontweight='bold')
        
        # Create subplots but DO NOT use imshow
        gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)
        
        # Original - just save as text/array info, no image display
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.text(0.5, 0.5, f'Original Image\nSize: {original_size}\nShape: {img_array.shape}', 
                ha='center', va='center', fontsize=10)
        ax1.set_title('Original Image Info', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # Heatmap info - no image
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.text(0.5, 0.5, f'Grad-CAM Heatmap\nShape: {cam.shape}\nMax: {cam.max():.3f}', 
                ha='center', va='center', fontsize=10)
        ax2.set_title('Grad-CAM Info', fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        # Overlay info - no image
        ax3 = fig.add_subplot(gs[0, 2])
        det_info = f"Detections: {len(detections)}\n"
        for det in detections[:3]:  # Show first 3
            det_info += f"Class {det['class']}: {det['confidence']:.2f}\n"
        ax3.text(0.5, 0.5, det_info, ha='center', va='center', fontsize=10)
        ax3.set_title('Detection Info', fontsize=12, fontweight='bold')
        ax3.axis('off')
        
        # SAVE TO FILE - NO DISPLAY - CRITICAL
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close(fig)  # CRITICAL: Close immediately
        del fig, ax1, ax2, ax3  # Clean up
        
        print(f"Grad-CAM saved: {save_path}")


# Generate Grad-CAM visualizations (AUTOMATICALLY RUNS AFTER TRAINING - saves to file only, no display)
print("=" * 100)
print(" Generating Grad-CAM Visualizations ".center(100, "="))
print("=" * 100)
print()

# Ensure run_dir is defined (from training cell)
if 'run_dir' not in globals():
    # Try to find the most recent run
    runs_base = Path("/kaggle/working/runs")
    if runs_base.exists():
        run_dirs = sorted([d for d in runs_base.iterdir() if d.is_dir() and d.name.startswith("yolo11m_seg_")], 
                          key=lambda x: x.stat().st_mtime, reverse=True)
        if run_dirs:
            run_dir = run_dirs[0]
            print(f"Using most recent run directory: {run_dir}")
        else:
            print("No run directory found, skipping Grad-CAM")
            run_dir = None
    else:
        print("No runs directory found, skipping Grad-CAM")
        run_dir = None
else:
    print(f"Using run directory from training: {run_dir}")

if run_dir and Path(run_dir).exists():
    val_images_dir = Path(f"{DATA_DIR}/images/val")
    if val_images_dir.exists():
        image_files = list(val_images_dir.glob("*.jpg"))[:4]  # Only 4 samples
        
        if image_files:
            print("Generating Grad-CAM visualizations (saved only, no display)...")
            
            best_model_path = Path(run_dir) / "weights" / "best.pt"
            if best_model_path.exists():
                best_model = YOLO(str(best_model_path))
                gradcam = GradCAM(best_model)
                
                gradcam_dir = Path(OUT_DIR) / "gradcam"
                gradcam_dir.mkdir(parents=True, exist_ok=True)
                
                for img_path in image_files:
                    # Run prediction first
                    pred_results = best_model.predict(
                        source=str(img_path),
                        conf=0.25,
                        save=False,
                        show=False,  # CRITICAL: No display
                        verbose=False
                    )
                    
                    if len(pred_results) > 0 and len(pred_results[0].boxes) > 0:
                        # Extract detections
                        detections = []
                        boxes = pred_results[0].boxes
                        for i in range(len(boxes)):
                            detections.append({
                                'class': int(boxes.cls[i].item()),
                                'confidence': float(boxes.conf[i].item())
                            })
                        
                        # Generate Grad-CAM
                        save_path = gradcam_dir / f"{img_path.stem}_gradcam.png"
                        gradcam.visualize(str(img_path), detections, save_path)
                
                print(f"Grad-CAM visualizations saved to: {gradcam_dir}")
                print(f"  Total visualizations: {len(list(gradcam_dir.glob('*.png')))}")
            else:
                print("Best model weights not found, skipping Grad-CAM")
                print(f"  Expected path: {best_model_path}")
        else:
            print("No validation images found for Grad-CAM")
    else:
        print("Validation images directory not found")
        print(f"  Expected path: {DATA_DIR}/images/val")
else:
    print("Run directory not found, skipping Grad-CAM")

print()
print("=" * 100)


In [ ]:
# Optional: Run predictions on validation set (SAVED ONLY, NO DISPLAY)
ENABLE_PREDICTIONS = False  # Set to True to enable

if ENABLE_PREDICTIONS:
    val_images_dir = Path(f"{DATA_DIR}/images/val")
    if val_images_dir.exists():
        # Get a few sample images
        image_files = list(val_images_dir.glob("*.jpg"))[:5]  # Only 5 samples
        
        if image_files:
            print("Running sample predictions (saved only, no display)...")
            
            # Load best model
            best_model_path = run_dir / "weights" / "best.pt"
            if best_model_path.exists():
                best_model = YOLO(str(best_model_path))
                
                # Run predictions - CRITICAL: show=False, save=True
                pred_results = best_model.predict(
                    source=[str(img) for img in image_files],
                    conf=0.25,
                    save=True,
                    show=False,  # CRITICAL: No display
                    project=OUT_DIR,
                    name="sample_predictions",
                    verbose=False
                )
                
                print(f"Sample predictions saved to: {OUT_DIR}/sample_predictions")
            else:
                print("Best model weights not found, skipping predictions")
        else:
            print("No validation images found")
    else:
        print("Validation images directory not found")
else:
    print("Sample predictions disabled (set ENABLE_PREDICTIONS = True to enable)")


## 11. Package Outputs for Download


In [ ]:
print("=" * 100)
print(" Packaging Outputs ".center(100, "="))
print("=" * 100)
print()

# Copy best weights to OUT_DIR
best_weights_src = run_dir / "weights" / "best.pt"
if best_weights_src.exists():
    best_weights_dst = Path(OUT_DIR) / "best.pt"
    shutil.copy(best_weights_src, best_weights_dst)
    print(f"Copied best weights to: {best_weights_dst}")
else:
    print("Best weights not found")

# Copy last weights
last_weights_src = run_dir / "weights" / "last.pt"
if last_weights_src.exists():
    last_weights_dst = Path(OUT_DIR) / "last.pt"
    shutil.copy(last_weights_src, last_weights_dst)
    print(f"Copied last weights to: {last_weights_dst}")

# Copy all checkpoints (periodic saves)
checkpoints_dir = run_dir / "weights"
if checkpoints_dir.exists():
    checkpoint_files = list(checkpoints_dir.glob("*.pt"))
    if checkpoint_files:
        checkpoints_dest = Path(OUT_DIR) / "checkpoints"
        checkpoints_dest.mkdir(parents=True, exist_ok=True)
        for ckpt in checkpoint_files:
            shutil.copy(ckpt, checkpoints_dest / ckpt.name)
        print(f"Copied {len(checkpoint_files)} checkpoint(s) to: {checkpoints_dest}")

# Copy training results CSV if exists
results_csv = run_dir / "results.csv"
if results_csv.exists():
    results_csv_dst = Path(OUT_DIR) / "results.csv"
    shutil.copy(results_csv, results_csv_dst)
    print(f"Copied results.csv to: {results_csv_dst}")

# Copy args.yaml (training configuration)
args_yaml = run_dir / "args.yaml"
if args_yaml.exists():
    args_yaml_dst = Path(OUT_DIR) / "args.yaml"
    shutil.copy(args_yaml, args_yaml_dst)
    print(f"Copied args.yaml to: {args_yaml_dst}")

# Copy training directory structure to OUT_DIR
training_outputs_dir = Path(OUT_DIR) / "training_outputs"
if run_dir.exists():
    shutil.copytree(run_dir, training_outputs_dir, dirs_exist_ok=True)
    print(f"Copied training outputs to: {training_outputs_dir}")

# Copy dataset.yaml to OUT_DIR (important for inference later)
dataset_yaml_src = Path(DATASET_YAML)
if dataset_yaml_src.exists():
    dataset_yaml_dst = Path(OUT_DIR) / "dataset.yaml"
    shutil.copy(dataset_yaml_src, dataset_yaml_dst)
    print(f"Copied dataset.yaml to: {dataset_yaml_dst}")
else:
    print("dataset.yaml not found")

# Copy Grad-CAM outputs if they exist
gradcam_dir_src = Path(OUT_DIR) / "gradcam"
if gradcam_dir_src.exists() and any(gradcam_dir_src.iterdir()):
    print(f"Grad-CAM outputs already in: {gradcam_dir_src}")
else:
    # Check if Grad-CAM was generated in a different location
    gradcam_check = Path(OUT_DIR) / "gradcam"
    if gradcam_check.exists():
        print(f"Grad-CAM outputs found in: {gradcam_check}")

# Copy training curves if they exist
training_curves = Path(OUT_DIR) / "training_curves.png"
if training_curves.exists():
    print(f"Training curves saved to: {training_curves}")

print()


In [ ]:
# Create ZIP archive for easy download
print("=" * 100)
print(" Creating ZIP Archive ".center(100, "="))
print("=" * 100)
print()

ZIP_PATH = "/kaggle/working/output_results.zip"

try:
    shutil.make_archive(
        base_name="/kaggle/working/output_results",
        format="zip",
        root_dir=OUT_DIR
    )
    
    # Verify ZIP was created
    if Path(ZIP_PATH).exists():
        zip_size = Path(ZIP_PATH).stat().st_size / (1024 * 1024)  # Size in MB
        print(f"\nCreated ZIP archive: {ZIP_PATH}")
        print(f"  ZIP size: {zip_size:.2f} MB")
        print(f"  All outputs saved to: {OUT_DIR}")
        print()
        print("=" * 100)
        print(" DOWNLOAD INSTRUCTIONS ".center(100, "="))
        print("=" * 100)
        print()
        print("To download outputs from Kaggle:")
        print("  1. Go to the Kaggle notebook output panel (right sidebar)")
        print("  2. Find 'output_results.zip' in the /kaggle/working/ folder")
        print("  3. Click the download icon next to the file")
        print()
        print("OR")
        print()
        print("  1. The entire /kaggle/working/outputs/ folder is available for download")
        print("  2. You can download individual files or the entire folder")
        print()
        print("Contents in ZIP:")
        print("  - best.pt, last.pt (model weights)")
        print("  - checkpoints/ (all periodic checkpoints)")
        print("  - exports/ (ONNX, TorchScript, TensorRT formats)")
        print("  - gradcam/ (Grad-CAM visualizations - saved to files)")
        print("  - dataset.yaml (for inference)")
        print("  - results.csv (training metrics)")
        print("  - training_curves.png (loss and mAP plots)")
        print("  - training_outputs/ (full training directory)")
        print()
    else:
        print(f"\nZIP file was not created at: {ZIP_PATH}")
        print(f"  Outputs are still available in: {OUT_DIR}")
        print()
except Exception as e:
    print(f"\nCould not create ZIP archive: {e}")
    print(f"  Outputs are still available in: {OUT_DIR}")
    print("  You can manually download the outputs folder from Kaggle")
    print()


In [ ]:
# Attempt to download outputs using Kaggle API (if available)
print("=" * 100)
print(" Automatic Download via Kaggle API ".center(100, "="))
print("=" * 100)
print()

# Note: Direct download from notebook to local machine is not possible
# Files in /kaggle/working/ are automatically available in Kaggle UI
# For automated downloads, use Kaggle API from your local machine

ZIP_PATH = "/kaggle/working/output_results.zip"  # Ensure ZIP_PATH is defined

try:
    import kaggle
    from kaggle.api.kaggle_api_extended import KaggleApi
    
    print("Kaggle API is available")
    print()
    print("To download outputs automatically from your local machine:")
    print("  1. Install Kaggle API: pip install kaggle")
    print("  2. Set up credentials: kaggle.json in ~/.kaggle/")
    print("  3. Run this command locally:")
    print()
    kernel_id = os.environ.get('KAGGLE_KERNEL_ID', 'YOUR_KERNEL_ID')
    print(f"     kaggle kernels output {kernel_id} -p ./downloads")
    print()
    print("OR download specific files:")
    print(f"     kaggle kernels output {kernel_id} -f output_results.zip")
    print()
    
except ImportError:
    print("Kaggle API not installed in this notebook")
    print()
    print("Files are automatically available for download in Kaggle UI:")
    print(f"  - {ZIP_PATH}")
    print(f"  - {OUT_DIR}/")
    print()
    print("To enable automatic downloads:")
    print("  1. Install: pip install kaggle")
    print("  2. Configure API credentials")
    print("  3. Use from your local machine (not from notebook)")
    print()

# Create download scripts for local use (both bash and Python)
kernel_id = os.environ.get('KAGGLE_KERNEL_ID', 'YOUR_KERNEL_ID')

# Bash script (Linux/Mac)
download_script_bash = f"""#!/bin/bash
# Download script for local machine (Linux/Mac)
# Run this AFTER the notebook completes on Kaggle

# Install Kaggle API first: pip install kaggle
# Configure: kaggle.json in ~/.kaggle/

KERNEL_ID="{kernel_id}"

echo "Downloading outputs from Kaggle kernel..."
kaggle kernels output $KERNEL_ID -p ./downloads

echo "Download complete! Check ./downloads/ folder"
"""

# Python script (Windows/Linux/Mac)
download_script_py = f"""#!/usr/bin/env python3
# Download script for local machine (Windows/Linux/Mac)
# Run this AFTER the notebook completes on Kaggle
# Usage: python download_outputs.py

import subprocess
import os

KERNEL_ID = "{kernel_id}"
DOWNLOAD_DIR = "./downloads"

print("Downloading outputs from Kaggle kernel...")
print(f"Kernel ID: {{KERNEL_ID}}")
print(f"Download to: {{DOWNLOAD_DIR}}")
print()

# Create download directory
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

try:
    # Download all outputs
    result = subprocess.run(
        ["kaggle", "kernels", "output", KERNEL_ID, "-p", DOWNLOAD_DIR],
        check=True,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    print("Download complete! Check ./downloads/ folder")
except subprocess.CalledProcessError as e:
    print(f"Error: {{e.stderr}}")
    print("Make sure:")
    print("  1. Kaggle API is installed: pip install kaggle")
    print("  2. Credentials are configured: kaggle.json in ~/.kaggle/")
    print("  3. Kernel ID is correct")
except FileNotFoundError:
    print("Kaggle CLI not found!")
    print("Install it: pip install kaggle")
"""

# Save scripts to outputs
download_script_bash_path = Path(OUT_DIR) / "download_outputs.sh"
download_script_py_path = Path(OUT_DIR) / "download_outputs.py"

with open(download_script_bash_path, 'w') as f:
    f.write(download_script_bash)
os.chmod(download_script_bash_path, 0o755)  # Make executable

with open(download_script_py_path, 'w') as f:
    f.write(download_script_py)
os.chmod(download_script_py_path, 0o755)  # Make executable

print(f"Created download scripts:")
print(f"  - {download_script_bash_path} (Linux/Mac)")
print(f"  - {download_script_py_path} (Windows/Linux/Mac)")
print()

print("=" * 100)
print(" IMPORTANT: Download Methods ".center(100, "="))
print("=" * 100)
print()
print("Method 1 (Easiest): Manual Download from Kaggle UI")
print("  - Files in /kaggle/working/ are automatically available")
print("  - Go to notebook output panel (right sidebar) → Download files")
print(f"  - Download: {ZIP_PATH}")
print()
print("Method 2: Kaggle API Command (from your local machine)")
print("  - Install: pip install kaggle")
print("  - Configure: kaggle.json in ~/.kaggle/ or C:\\Users\\YourName\\.kaggle\\")
print(f"  - Run: kaggle kernels output {kernel_id} -f output_results.zip")
print()
print("Method 3: Use the download scripts (from your local machine)")
print(f"  - Linux/Mac: bash {download_script_bash_path.name}")
print(f"  - Windows: python {download_script_py_path.name}")
print("  - Scripts are saved in the outputs folder")
print()
print("NOTE: Direct download from notebook code is NOT possible")
print("      (notebook runs on Kaggle servers, not your machine)")
print("      Use one of the methods above from your local machine")
print()
print("=" * 100)
print()


In [ ]:
# Final summary
print("=" * 100)
print(" Training Summary ".center(100, "="))
print("=" * 100)
print()
print(f"Model: {MODEL_CONFIG['model']}")
print(f"Dataset: {DATA_DIR}")
print(f"Training outputs: {run_dir}")
print(f"All outputs packaged: {OUT_DIR}")
print(f"ZIP file: /kaggle/working/output_results.zip")
print()
print("Contents in ZIP:")
print("  - best.pt (best model weights)")
print("  - last.pt (last checkpoint)")
print("  - checkpoints/ (all periodic checkpoints)")
print("  - exports/ (ONNX, TorchScript, TensorRT formats)")
print("  - gradcam/ (Grad-CAM visualizations - all saved to files)")
print("  - dataset.yaml (for inference)")
print("  - results.csv (training metrics)")
print("  - training_curves.png (loss and mAP plots)")
print("  - training_outputs/ (full training directory)")
print()
print("Next steps:")
print("  1. Download output_results.zip from Kaggle working folder")
print("  2. OR download the entire /kaggle/working/outputs/ folder")
print("  3. Extract to view training curves, weights, Grad-CAM, and results")
print("  4. Use best.pt or exported formats for inference")
print()
print("=" * 100)


In [ ]:
# Final summary
print("=" * 100)
print(" Training Summary ".center(100, "="))
print("=" * 100)
print()
print(f"Model: {MODEL_CONFIG['model']}")
print(f"Dataset: {DATA_DIR}")
print(f"Training outputs: {run_dir}")
print(f"All outputs packaged: {OUT_DIR}")
print(f"ZIP file: /kaggle/working/output_results.zip")
print()
print("Contents in ZIP:")
print("  - best.pt (best model weights)")
print("  - last.pt (last checkpoint)")
print("  - checkpoints/ (all periodic checkpoints)")
print("  - exports/ (ONNX, TorchScript, TensorRT formats)")
print("  - dataset.yaml (for inference)")
print("  - results.csv (training metrics)")
print("  - training_curves.png (loss and mAP plots)")
print("  - training_outputs/ (full training directory)")
print()
print("Next steps:")
print("  1. Download the ZIP file from Kaggle outputs")
print("  2. Extract to view training curves, weights, and results")
print("  3. Use best.pt or exported formats for inference")
print()
print("=" * 100)


In [ ]:
# Notebook complete - all outputs saved and zipped


In [ ]:
# Notebook complete - all outputs saved and zipped


In [ ]:
# Notebook complete - all outputs saved and zipped


In [ ]:
# Notebook complete - all outputs saved and zipped
